# papers

https://github.com/jdh-observer/...

-------------------------------------------------------------------------------
https://github.com/jdh-observer/...

--------------------------------------------------------------------------------
https://github.com/jdh-observer/... 



In [3]:
import os

venv = os.path.expanduser("~/venvs/notebookpdf")
!python3 -m venv "$venv"

!"$venv/bin/python" -m pip install --upgrade pip

In [4]:
from __future__ import annotations

import json
import re
import urllib.request
from html import unescape
from html.parser import HTMLParser
from pathlib import Path
from urllib.parse import urlparse


# =========================================================
# SETTINGS
# =========================================================
JDH_ROOT = Path(".")
BASE_OUTPUT_ROOT = JDH_ROOT / "review_outputs" / "paper_reviews"

NOTEBOOK_SOURCES = [
    {
        "label": "author1",
        "paper_dir": BASE_OUTPUT_ROOT / "paper_001",
        "repo_url": "https://github.com/...",
        "notebook_url": "https://github.com/.../...",
    },
    {
        "label": "author2",
        "paper_dir": BASE_OUTPUT_ROOT / "paper_002",
        "repo_url": "https://github.com/...",
        "notebook_url": "https://github.com/.../...",
    },
    {
        "label": "author3",
        "paper_dir": BASE_OUTPUT_ROOT / "paper_003",
        "repo_url": "https://github.com/...",
        "notebook_url": "https://github.com/...",
    },
]

OUTPUT_TXT_NAME = "article_extracted_reader_text_with_code.txt"
SAVE_RAW_IPYNB = False
RAW_IPYNB_NAME = "article_downloaded.ipynb"

INCLUDE_CODE_CELLS = True
INCLUDE_CODE_OUTPUTS = True


# =========================================================
# HELPERS
# =========================================================
def clean_text(text) -> str:
    if text is None:
        return ""
    text = str(text).replace("\r\n", "\n").replace("\r", "\n").replace("\x00", " ")
    text = text.replace("\xa0", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n[ \t]+\n", "\n\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def pretty_path(path: Path) -> str:
    path = Path(path).resolve()
    try:
        return path.relative_to(Path.cwd().resolve()).as_posix()
    except Exception:
        return path.as_posix()


def github_blob_to_raw(url: str) -> str:
    parsed = urlparse(url)
    parts = [p for p in parsed.path.split("/") if p]
    if len(parts) < 5 or parts[2] != "blob":
        raise ValueError(f"Unsupported GitHub blob URL: {url}")

    owner = parts[0]
    repo = parts[1]
    branch = parts[3]
    rest = "/".join(parts[4:])

    return f"https://raw.githubusercontent.com/{owner}/{repo}/{branch}/{rest}"


def download_text(url: str) -> str:
    req = urllib.request.Request(
        url,
        headers={
            "User-Agent": "Mozilla/5.0",
            "Accept": "*/*",
        },
    )
    with urllib.request.urlopen(req, timeout=120) as resp:
        data = resp.read()
    return data.decode("utf-8", errors="replace")


def join_source(source) -> str:
    if source is None:
        return ""
    if isinstance(source, list):
        return "".join(str(x) for x in source)
    return str(source)


def normalize_block(text: str) -> str:
    text = clean_text(text).lower()
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def append_block(blocks: list[str], seen: set[str], text: str) -> None:
    text = clean_text(text)
    if not text:
        return

    key = normalize_block(text)
    if not key:
        return

    if key in seen:
        return

    seen.add(key)
    blocks.append(text)


def looks_like_useless_output(text: str) -> bool:
    t = clean_text(text)
    if not t:
        return True

    low = t.lower()

    # skip figure placeholders and trivial repr noise
    if re.fullmatch(r"<figure size .*?>", low):
        return True
    if re.fullmatch(r"figure\(\d+x\d+\)", low):
        return True
    if low in {"none", "nan"}:
        return True

    return False


class ReaderHTMLExtractor(HTMLParser):
    def __init__(self):
        super().__init__()
        self.lines: list[str] = []
        self.current_text: list[str] = []

        self.in_table = False
        self.in_row = False
        self.in_cell = False

        self.table_rows: list[list[str]] = []
        self.current_row: list[str] = []
        self.current_cell: list[str] = []

    def _flush_text(self):
        text = clean_text("".join(self.current_text))
        if text:
            self.lines.append(text)
        self.current_text = []

    def _flush_table(self):
        if not self.table_rows:
            return

        rendered_rows = []
        for row in self.table_rows:
            row = [clean_text(cell) for cell in row]
            row = [cell for cell in row if cell]
            if row:
                rendered_rows.append(" | ".join(row))

        if rendered_rows:
            self.lines.append("\n".join(rendered_rows))

        self.table_rows = []

    def handle_starttag(self, tag, attrs):
        tag = tag.lower()

        if tag in {"p", "div", "section", "article", "header", "footer", "h1", "h2", "h3", "h4", "h5", "h6"}:
            self._flush_text()

        elif tag == "br":
            self._flush_text()

        elif tag in {"ul", "ol"}:
            self._flush_text()

        elif tag == "li":
            self._flush_text()
            self.current_text.append("- ")

        elif tag == "table":
            self._flush_text()
            self.in_table = True
            self.table_rows = []

        elif tag == "tr":
            self.in_row = True
            self.current_row = []

        elif tag in {"td", "th"}:
            self.in_cell = True
            self.current_cell = []

    def handle_endtag(self, tag):
        tag = tag.lower()

        if tag in {"p", "div", "section", "article", "header", "footer", "h1", "h2", "h3", "h4", "h5", "h6"}:
            self._flush_text()

        elif tag == "li":
            self._flush_text()

        elif tag in {"td", "th"}:
            cell_text = clean_text("".join(self.current_cell))
            self.current_row.append(cell_text)
            self.current_cell = []
            self.in_cell = False

        elif tag == "tr":
            if self.current_row:
                self.table_rows.append(self.current_row)
            self.current_row = []
            self.in_row = False

        elif tag == "table":
            self._flush_table()
            self.in_table = False

    def handle_data(self, data):
        if not data:
            return

        if self.in_cell:
            self.current_cell.append(data)
        else:
            self.current_text.append(data)

    def get_text(self) -> str:
        self._flush_text()
        self._flush_table()
        text = "\n\n".join(x for x in self.lines if clean_text(x))
        text = text.replace("\xa0", " ")
        text = re.sub(r"[ \t]+", " ", text)
        text = re.sub(r"\n{3,}", "\n\n", text)
        return text.strip()


def html_to_text(html: str) -> str:
    parser = ReaderHTMLExtractor()
    parser.feed(unescape(html or ""))
    parser.close()
    return clean_text(parser.get_text())


def markdown_to_text(md: str) -> str:
    md = join_source(md)
    md = md.replace("\r\n", "\n").replace("\r", "\n")
    md = unescape(md)

    # Remove fenced code blocks in markdown cells only
    md = re.sub(r"```.*?```", " ", md, flags=re.DOTALL)

    # Images: keep alt text
    md = re.sub(r"!\[([^\]]*)\]\([^)]+\)", r"\1", md)

    # Links: keep link text
    md = re.sub(r"\[([^\]]+)\]\([^)]+\)", r"\1", md)

    # Inline HTML
    if "<" in md and ">" in md:
        md = html_to_text(md)

    # headings / lists / quotes
    md = re.sub(r"(?m)^\s{0,3}#{1,6}\s*", "", md)
    md = re.sub(r"(?m)^\s{0,3}>\s?", "", md)
    md = re.sub(r"(?m)^\s*[-*+]\s+", "- ", md)
    md = re.sub(r"(?m)^\s*\d+\.\s+", "- ", md)

    # emphasis markers
    md = md.replace("**", "")
    md = md.replace("__", "")
    md = md.replace("*", "")
    md = md.replace("_", "")
    md = md.replace("`", "")

    md = re.sub(r"[ \t]+", " ", md)
    md = re.sub(r"\n{3,}", "\n\n", md)

    return clean_text(md)


def json_to_text(data) -> str:
    try:
        return clean_text(json.dumps(data, ensure_ascii=False, indent=2))
    except Exception:
        return clean_text(str(data))


def render_code_source(code: str) -> str:
    code = join_source(code).rstrip()
    if not clean_text(code):
        return ""
    return "[CODE]\n" + code


def render_output_text(output: dict) -> str:
    out_type = output.get("output_type", "")

    if out_type == "stream":
        text = clean_text(join_source(output.get("text", "")))
        return "" if looks_like_useless_output(text) else text

    if out_type == "error":
        traceback_lines = output.get("traceback", [])
        if traceback_lines:
            text = clean_text("\n".join(str(x) for x in traceback_lines))
            return "" if looks_like_useless_output(text) else text

        text = clean_text(f"{output.get('ename', '')}: {output.get('evalue', '')}")
        return "" if looks_like_useless_output(text) else text

    if out_type in ("display_data", "execute_result"):
        data = output.get("data", {})

        # Prefer html first because tables render there best
        if "text/html" in data:
            text = html_to_text(join_source(data["text/html"]))
            if text and not looks_like_useless_output(text):
                return text

        if "text/markdown" in data:
            text = markdown_to_text(data["text/markdown"])
            if text and not looks_like_useless_output(text):
                return text

        if "text/plain" in data:
            text = clean_text(join_source(data["text/plain"]))
            if text and not looks_like_useless_output(text):
                return text

        if "application/json" in data:
            text = json_to_text(data["application/json"])
            if text and not looks_like_useless_output(text):
                return text

    return ""


def notebook_to_reader_text_with_code(notebook_obj: dict, source_url: str, repo_url: str) -> str:
    blocks: list[str] = []
    seen: set[str] = set()

    append_block(blocks, seen, f"Source repo: {repo_url}")
    append_block(blocks, seen, f"Notebook: {source_url}")

    for cell in notebook_obj.get("cells", []):
        cell_type = str(cell.get("cell_type", ""))

        if cell_type == "markdown":
            text = markdown_to_text(cell.get("source", ""))
            append_block(blocks, seen, text)

        elif cell_type == "raw":
            text = clean_text(join_source(cell.get("source", "")))
            append_block(blocks, seen, text)

        elif cell_type == "code":
            if INCLUDE_CODE_CELLS:
                code_text = render_code_source(cell.get("source", ""))
                append_block(blocks, seen, code_text)

            if INCLUDE_CODE_OUTPUTS:
                for output in cell.get("outputs", []):
                    out_text = render_output_text(output)
                    if out_text:
                        append_block(blocks, seen, out_text)

    text = "\n\n".join(blocks)
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip() + "\n"


# =========================================================
# RUN
# =========================================================
BASE_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

written = []
errors = []

for item in NOTEBOOK_SOURCES:
    label = item["label"]
    paper_dir = Path(item["paper_dir"])
    repo_url = item["repo_url"]
    notebook_url = item["notebook_url"]

    try:
        raw_url = github_blob_to_raw(notebook_url)
        raw_nb_text = download_text(raw_url)
        nb = json.loads(raw_nb_text)

        extracted_text = notebook_to_reader_text_with_code(
            notebook_obj=nb,
            source_url=notebook_url,
            repo_url=repo_url,
        )

        paper_dir.mkdir(parents=True, exist_ok=True)

        txt_path = paper_dir / OUTPUT_TXT_NAME
        txt_path.write_text(extracted_text, encoding="utf-8")

        if SAVE_RAW_IPYNB:
            ipynb_path = paper_dir / RAW_IPYNB_NAME
            ipynb_path.write_text(raw_nb_text, encoding="utf-8")
        else:
            ipynb_path = None

        written.append({
            "label": label,
            "paper_dir": pretty_path(paper_dir),
            "txt_file": pretty_path(txt_path),
            "raw_ipynb_file": pretty_path(ipynb_path) if ipynb_path else "",
            "chars": len(extracted_text),
            "cells": len(nb.get("cells", [])),
        })

        print(f"[OK] {label} -> {pretty_path(txt_path)}")

    except Exception as e:
        errors.append({
            "label": label,
            "paper_dir": pretty_path(paper_dir),
            "error": str(e),
        })
        print(f"[FAILED] {label} :: {e}")

print("\nDone.")
print("Written:", len(written))
print("Errors :", len(errors))

if written:
    print("\nOutputs:")
    for x in written:
        print(" -", x["txt_file"])

[OK] author1 -> review_outputs/paper_reviews/paper_001/article_extracted_reader_text_with_code.txt
[OK] author2 -> review_outputs/paper_reviews/paper_002/article_extracted_reader_text_with_code.txt
[OK] author3 -> review_outputs/paper_reviews/paper_003/article_extracted_reader_text_with_code.txt

Done.
Written: 3
Errors : 0

Outputs:
 - review_outputs/paper_reviews/paper_001/article_extracted_reader_text_with_code.txt
 - review_outputs/paper_reviews/paper_002/article_extracted_reader_text_with_code.txt
 - review_outputs/paper_reviews/paper_003/article_extracted_reader_text_with_code.txt
